In [1]:
# %%
import pandas as pd 
import numpy as np
# import matplotlib.pyplot as plt 
# import seaborn as sns
import torch
from functools import partial
# from transformers import AutoTokenizer, AutoModel, AutoModelForCausalLM

import sys, os
project_root = os.path.abspath(os.path.join(os.getcwd(), "../.."))
if project_root not in sys.path:
    sys.path.append(project_root)

import asyncio
from lightrag.utils import setup_logger, EmbeddingFunc
from lightrag.llm.hf import hf_embed
from lightrag import LightRAG, QueryParam
from lightrag.llm.openai import gpt_4o_mini_complete, gpt_4o_complete, openai_embed, openai_complete
from lightrag.utils import setup_logger

# %%
setup_logger("lightrag", level="INFO")

WORKING_DIR = "./rag_storage"
if not os.path.exists(WORKING_DIR):
    os.mkdir(WORKING_DIR)

In [2]:
async def hf_model_complete(prompt: str, system_prompt=None, history_messages=[], **kwargs):
    device = model.device
    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    messages.extend(history_messages)
    messages.append({"role": "user", "content": prompt})
    
    # Use chat template to wrap the complex LightRAG instructions
    tokenized_chat = llm_tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt")
    inputs = {"input_ids": tokenized_chat.to(device)}

    outputs = model.generate(
        **inputs,
        max_new_tokens=512,
        temperature=0.1,
        do_sample=True,
        pad_token_id=llm_tokenizer.eos_token_id
    )

    decoded = llm_tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)
    return decoded.strip()

In [3]:
async def bio_mistral_complete(prompt, system_prompt=None, history_messages=None, **kwargs):
    # Update kwargs to be more strict for extraction
    kwargs.update({
        "temperature": 0.1,      # Be deterministic
        "top_p": 0.95,
        "max_tokens": 1024,      # Ensure it doesn't cut off mid-list
    })

    if system_prompt:
        # Prepend system prompt and emphasize the separator format
        prompt = (
            f"SYSTEM INSTRUCTIONS:\n{system_prompt}\n\n"
            f"IMPORTANT: You MUST use the exact separator <SEP> between fields and "
            f"END the response with the ############################# delimiter.\n\n"
            f"USER INPUT:\n{prompt}"
        )
        system_prompt = None
        
    return await openai_complete(
        prompt,
        system_prompt=system_prompt,
        history_messages=history_messages,
        **kwargs
    )

In [4]:
# Using vLLM

rag = LightRAG(
    working_dir=WORKING_DIR,
    llm_model_func=bio_mistral_complete,
    llm_model_name="BioMistral/BioMistral-7B-AWQ-QGS128-W4-GEMM",
    llm_model_max_async=4,
    llm_model_kwargs={
        "base_url": "http://127.0.0.1:8080/v1", 
        "api_key": "none" 
    },
    chunk_token_size=1200,
    entity_extract_max_gleaning=0,
    default_embedding_timeout=120,
    
    embedding_func=EmbeddingFunc(
        embedding_dim=768, 
        max_token_size=8192,
        func=partial(
            openai_embed.func,
            base_url="http://127.0.0.1:8085/v1",  # Updated port
            api_key="none",
            model="nomic-ai/nomic-embed-text-v1.5"
        )
    )
)

INFO: [] Created new empty graph file: ./rag_storage/graph_chunk_entity_relation.graphml
INFO:nano-vectordb:Init {'embedding_dim': 768, 'metric': 'cosine', 'storage_file': './rag_storage/vdb_entities.json'} 0 data
INFO:nano-vectordb:Init {'embedding_dim': 768, 'metric': 'cosine', 'storage_file': './rag_storage/vdb_relationships.json'} 0 data
INFO:nano-vectordb:Init {'embedding_dim': 768, 'metric': 'cosine', 'storage_file': './rag_storage/vdb_chunks.json'} 0 data


In [5]:
await rag.initialize_storages()

INFO: [] Process 504594 KV load full_docs with 1 records
INFO: [] Process 504594 KV load text_chunks with 0 records
INFO: [] Process 504594 KV load full_entities with 0 records
INFO: [] Process 504594 KV load full_relations with 0 records
INFO: [] Process 504594 KV load entity_chunks with 0 records
INFO: [] Process 504594 KV load relation_chunks with 0 records
INFO: [] Process 504594 KV load llm_response_cache with 0 records
INFO: [] Process 504594 doc status load doc_status with 3 records


In [6]:
data_dir = os.path.join(project_root, 'biomed-rag', 'data', 'external', 'medqa')
sample_textbook = os.path.join(data_dir, 'textbooks', 'Anatomy_Gray.txt')

with open(sample_textbook, 'r') as f:
    text = f.read()

In [7]:
await rag.ainsert(text)

INFO: Created 1 duplicate document records with track_id: insert_20260311_154555_1d7bb594
INFO: Preserving 3 failed document entries for manual review
INFO: Reset 1 documents from PROCESSING/FAILED to PENDING status
INFO: Processing 1 document(s)
INFO: Extracting stage 1/1: unknown_source
INFO: Processing d-id: doc-21250d6571086cc53d058350568a0609
INFO: Embedding func: 8 new workers initialized (Timeouts: Func: 120s, Worker: 240s, Health Check: 255s)
INFO:openai._base_client:Retrying request to /embeddings in 0.376551 seconds
INFO:openai._base_client:Retrying request to /embeddings in 0.456182 seconds
INFO:openai._base_client:Retrying request to /embeddings in 0.380232 seconds
INFO:openai._base_client:Retrying request to /embeddings in 0.383040 seconds
INFO:openai._base_client:Retrying request to /embeddings in 0.473778 seconds
INFO:openai._base_client:Retrying request to /embeddings in 0.383415 seconds
INFO:openai._base_client:Retrying request to /embeddings in 0.431858 seconds
INFO:o

CancelledError: 

INFO:openai._base_client:Retrying request to /embeddings in 0.381787 seconds
INFO:openai._base_client:Retrying request to /embeddings in 0.479076 seconds
INFO:openai._base_client:Retrying request to /embeddings in 0.417576 seconds
INFO:openai._base_client:Retrying request to /embeddings in 0.376273 seconds
INFO:openai._base_client:Retrying request to /embeddings in 0.498345 seconds
INFO:openai._base_client:Retrying request to /embeddings in 0.491424 seconds
INFO:openai._base_client:Retrying request to /embeddings in 0.466896 seconds
INFO:openai._base_client:Retrying request to /embeddings in 0.439942 seconds
INFO:openai._base_client:Retrying request to /embeddings in 0.882591 seconds
INFO:openai._base_client:Retrying request to /embeddings in 0.900800 seconds
INFO:openai._base_client:Retrying request to /embeddings in 0.964007 seconds
INFO:openai._base_client:Retrying request to /embeddings in 0.765415 seconds
INFO:openai._base_client:Retrying request to /embeddings in 0.768365 seconds

In [ ]:

# #### Test Retrieve

# query_1 = '''
# Compare and contrast the regional and systemic approaches to studying anatomy. Under what specific circumstances would one approach be preferred over the other?
# '''

# query_2 = '''
# How does the etymological origin of the word 'anatomy' relate to the primary techniques used to study it, and what modern alternatives have been introduced to supplement traditional methods?
# '''

# query_3 = '''
# What is anatomy?
# '''

# mode = "global"

# response = await rag.aquery(
#         query_3,
#         param=QueryParam(mode=mode)
#     )
